# Stress Score Prediction V3 — ExtraTrees

V2의 누수 없는 전처리와 행 단위 파생변수를 유지하고, 동일한 5-Fold 검증에서 가장 좋은 ExtraTrees 모델을 적용한 버전입니다.


## 0. Import & Data Load


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score


In [2]:
DATA_DIR = Path('/Users/bitsaem/Desktop/stress_project/open')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (3000, 18)
Test shape: (3000, 17)


## 1. Data Split


In [3]:
TARGET = 'stress_score'
ID_COL = 'ID'

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].copy()

X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL].copy()


## 2. Row-level Feature Engineering


In [4]:
def add_features(df):
    """다른 행이나 데이터셋 통계를 사용하지 않는 행 단위 파생변수 생성 함수."""
    df = df.copy()

    height_m = df['height'] / 100
    sbp = df['systolic_blood_pressure']
    dbp = df['diastolic_blood_pressure']

    df['bmi'] = df['weight'] / (height_m ** 2)
    df['pulse_pressure'] = sbp - dbp
    df['mean_arterial_pressure'] = (sbp + 2 * dbp) / 3
    df['blood_pressure_ratio'] = sbp / (dbp + 1e-6)
    df['cholesterol_glucose_ratio'] = (
        df['cholesterol'] / (df['glucose'] + 1e-6)
    )
    df['missing_count'] = df.isna().sum(axis=1)

    return df


X_v3 = add_features(X)
X_test_v3 = add_features(X_test)

print('Original shape:', X.shape)
print('V3 shape:', X_v3.shape)


Original shape: (3000, 16)
V3 shape: (3000, 22)


## 3. Leakage-safe Preprocessing


In [5]:
categorical_cols = X_v3.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols = X_v3.select_dtypes(
    include=['number']
).columns.tolist()

print('Categorical:', categorical_cols)
print('Numerical:', numerical_cols)


Categorical: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']
Numerical: ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'cholesterol_glucose_ratio', 'missing_count']


In [6]:
categorical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='constant',
            fill_value='missing'
        )
    ),
    (
        'encoder',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        )
    )
])

numerical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='median',
            add_indicator=True
        )
    )
])

preprocessor_v3 = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_cols),
    ('numerical', numerical_pipeline, numerical_cols)
])


## 4. ExtraTrees Model & Cross-validation


In [7]:
model_v3 = ExtraTreesRegressor(
    n_estimators=600,
    min_samples_leaf=1,
    max_features=0.6,
    random_state=42,
    n_jobs=-1
)

pipeline_v3 = Pipeline([
    ('preprocessor', preprocessor_v3),
    ('model', model_v3)
])


In [8]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

v3_mae = -cross_val_score(
    pipeline_v3,
    X_v3,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

V1_CV_MAE = 0.211783
V2_CV_MAE = 0.208324

print('V3 Fold MAE:', np.round(v3_mae, 6))
print('V1 Mean MAE:', V1_CV_MAE)
print('V2 Mean MAE:', V2_CV_MAE)
print('V3 Mean MAE:', round(v3_mae.mean(), 6))
print('V3 vs V2:', round(v3_mae.mean() - V2_CV_MAE, 6))


V3 Fold MAE: [0.163834 0.17733  0.175906 0.178831 0.177834]
V1 Mean MAE: 0.211783
V2 Mean MAE: 0.208324
V3 Mean MAE: 0.174747
V3 vs V2: -0.033577


## 5. Final Training & Prediction


In [9]:
pipeline_v3.fit(X_v3, y)

pred_v3 = pipeline_v3.predict(X_test_v3)
pred_v3 = np.clip(pred_v3, 0, 1)

print('Prediction shape:', pred_v3.shape)
print('Prediction range:', pred_v3.min(), pred_v3.max())


Prediction shape: (3000,)
Prediction range: 0.009749999999999998 0.9909833333333335


## 6. Submission


In [10]:
submission_v3 = submission.copy()
submission_v3[ID_COL] = test_ids
submission_v3[TARGET] = pred_v3

output_path_v3 = DATA_DIR.parent / 'submit_v3_extratrees.csv'
submission_v3.to_csv(output_path_v3, index=False)

print('Saved:', output_path_v3)
submission_v3.head()


Saved: /Users/bitsaem/Desktop/stress_project/submit_v3_extratrees.csv


,ID,stress_score
0,TEST_0000,0.568650
1,TEST_0001,0.928567
2,TEST_0002,0.355900
3,TEST_0003,0.440633
4,TEST_0004,0.527817
